In [ ]:
2+2

4

In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
def get_weather(city: str) -> str:
    """Get the weather for the city"""
    return f'The weather is {city} is sunny'

agent=create_agent(model='gpt-3.5-turbo',tools=[get_weather],
             system_prompt='you are helpful assistant')

result=agent.invoke({"messages": [{"role":"user", 'content':'what is the weather in Indore city?'}]})
print(result['messages'][-1].content)

The weather in Indore city is sunny.


In [12]:
#MOdel intergration
#Summarization mddleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model='gpt-3.5-turbo',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini',
            trigger=("messages",10),
            keep=("messages",3)
        )
    ]
)

config = {'configurable':{'thread_id':"test-1"}}
que=['what is 1+3', 'what is 3*3']

for q in que:
    response=agent.invoke({'messages':[HumanMessage(content=q)]},config)
    print(f'Message: {response['messages'][-1].content}')

Message: 1 + 3 equals 4.
Message: 3 * 3 equals 9.


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """search hotels - return long response to use more tokens"""
    return f"""Hotel in {city}:
    1. Grand hotel - 5 star, $350/night
    2. City in - 4-star, $180/night"""

agent=create_agent(model='gpt-3.5-turbo',tools=[search_hotels],
             checkpointer=InMemorySaver(),
             middleware=[SummarizationMiddleware(model='gpt-4o-mini',
                                                trigger=("tokens",500),keep=("tokens",200))])
config = {'configurable':{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

cities = ['Paris','London','Tokyo','New York']

for city in cities:
    response=agent.invoke({'messages':[HumanMessage(content=f'Find hotels in {city}')]},config=config)
    tokens=count_tokens(response['messages'])
                          
    print(f'{city}: ~{tokens} tokens,{len(response['messages'])} messages')   
    print(f'{(response['messages'][-1].content)}')



Paris: ~70 tokens,4 messages
I found some hotels in Paris for you:
1. Grand hotel - 5 star, $350/night
2. City inn - 4-star, $180/night

Would you like more options or details about any specific hotel?
London: ~142 tokens,8 messages
I found some hotels in London for you:
1. Grand hotel - 5 star, $350/night
2. City inn - 4-star, $180/night

Would you like more options or details about any specific hotel?
Tokyo: ~235 tokens,13 messages
I found some hotels in Tokyo for you:
1. Grand hotel - 5 star, $350/night
2. City inn - 4-star, $180/night

Would you like more options or details about any specific hotel?
New York: ~378 tokens,7 messages
I found some hotels in New York for you:
1. Grand hotel - 5 star, $350/night
2. City Inn - 4-star, $180/night

These are the options available in New York. Let me know if you need more information or options!


In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its IDs"""
    return f'Email content for ID:{email_id}'
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send and email"""
    return f"Email sent to {recipient} with subject '{subject}'"

agent=create_agent(model='gpt-4o-mini',tools=[read_email_tool,send_email_tool],
                   checkpointer=InMemorySaver(),
                   middleware=[HumanInTheLoopMiddleware(interrupt_on=
                                                        {"send_email_tool":{"allowed_decisions":['approve','edit','reject']},
                                                                      "read_email_tool":False})]
                                                                      )
config = {'configurable':{'thread_id':'test-approve'}}
result=agent.invoke({'messages':[HumanMessage(content='Send email to john@example.com with subject "Hello" and body "How are you?"')]}, config=config)
print(result['messages'][-1].content)

from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving....")
    result = agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)
    print(f'Result: {result['messages'][-1].content}')


Paused! Approving....
Result: Email has been sent to john@example.com with the subject "Hello" and the body "How are you?"


In [28]:
#Structured Output

from langchain.chat_models import init_chat_model
model=init_chat_model("groq:llama-3.1-8b-instant")
model=init_chat_model("openai:gpt-3.5-turbo")
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description='This year the movie eas released')
    director:str=Field(description='The director of the movie')
    rating:float=Field(description='The movie rating out of 10')
    
movie_with_structure=model.with_structured_output(Movie)
movie_with_structure

result=model.invoke("Please details about movie Inception")
print(result.content)

c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2480: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


Inception is a 2010 science fiction action film written and directed by Christopher Nolan. The film stars Leonardo DiCaprio as Dom Cobb, a professional thief who steals information by entering the dreams of his targets. Cobb is offered a chance to have his criminal history erased if he can successfully plant an idea into the mind of a CEO's son.

The film explores the concept of dream-sharing technology and the ability to manipulate dreams. Cobb assembles a team of experts, including Arthur (Joseph Gordon-Levitt) and Ariadne (Ellen Page), to carry out the dangerous mission of inception. However, Cobb's troubled past and his guilt over his wife's death begin to interfere with the mission, leading to a series of escalating challenges and threats.

Inception is known for its complex and mind-bending narrative, stunning visual effects, and intense action sequences. The film received critical acclaim for its originality, storytelling, and performances. It was a box office success, grossing 

In [29]:
movie_with_structure.invoke("Please details about movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [30]:
###  Message output along side parsed structure

In [36]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description='The year the movie was released')
    director:str=Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movie's rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)
response=model_with_structure.invoke("Please provide details about the Movie Inception")
response

c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2480: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


{'raw': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 103, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EbfP37yz5q8tB03nlC6qnrSxUd2qx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a16207-2077-7495-a003-f808f6147281-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}, 'id': 'call_QqxrUCXjQLOLB9hwx9Ei8TPM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 103, 'output_tokens': 22, 'total_tokens': 125, 'input_token_details': {'audio': 0, 'cache_r

In [37]:
#Nested structure

from pydantic import BaseModel, Field
class Actor(BaseModel):
    name: str
    role: str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Avengers")
response

c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2480: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


MovieDetails(title='The Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark/Iron Man'), Actor(name='Chris Evans', role='Steve Rogers/Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner/The Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff/Black Widow'), Actor(name='Jeremy Renner', role='Clint Barton/Hawkeye')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=220.0)

In [41]:
#TypedDict

from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ... , "The title of the movie"]
    year: Annotated[int, ...,"The year the movie released"]
    director:Annotated[str, ..., "The directed name"]
    rating: Annotated[float, ...,"The movie rating"]

model_with_typedict=model.with_structured_output(MovieDict)
response=model_with_typedict.invoke("Please provide the details of movie Don")
response

c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2480: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


{'title': 'Don', 'year': 2006, 'director': 'Farhan Akhtar', 'rating': 7.2}

In [43]:

class Actor(TypedDict):
    name: str
    role: str
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Earning in INR")

model_with_structure = model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Please provide details on Movie Dhoom-2")
response

c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2480: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


{'title': 'Dhoom-2',
 'year': 2006,
 'cast': [{'name': 'Hrithik Roshan', 'role': 'Aryan Singh/Mr.A'},
  {'name': 'Abhishek Bachchan', 'role': 'A.C.P. Jai Dixit'},
  {'name': 'Aishwarya Rai', 'role': 'Sunehri'},
  {'name': 'Uday Chopra', 'role': 'Ali Akbar Fateh Khan'}],
 'genres': ['Action', 'Thriller'],
 'budget': 35000000}

In [45]:
#DATACLASS

from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:str=Field(description="The phone number of the person")

agent=create_agent(model='gpt-3.5-turbo',
                   response_format=ContactInfo)
result=agent.invoke({"messages":[{"role":"user","content":"Extract the contact info from John Doe, john@example.com, 535435436436"}]})
print(result)

{'messages': [HumanMessage(content='Extract the contact info from John Doe, john@example.com, 535435436436', additional_kwargs={}, response_metadata={}, id='85c17164-068c-44b4-92db-f477f17bf7f3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 93, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Ebg5oWpIQKPcH5IZJLROzAvTP2ing', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1622f-9504-7055-a9fd-c121e5704b80-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '535435436436'}, 'id': 'call_4p2x6e

In [46]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='535435436436')

In [49]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo():
    """Contact info for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(model="gpt-3.5-turbo",response_format=ContactInfo)

result=agent.invoke({"messages":[{"role":"user","content":"Extract the contact info from John Doe, john@example.com, 535435436436"}]})
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='535435436436')

In [51]:
from langchain.chat_models import init_chat_model
model=init_chat_model('openai:gpt-3.5-turbo')
result=model.invoke("What is deep agent ?")
print(result.content)

A deep agent is an artificial intelligence system that uses deep learning algorithms to make decisions and take actions in a given environment. Deep agents are often used in areas such as robotics, autonomous vehicles, gaming, and other complex tasks where traditional algorithms may not be effective. These agents are able to learn from large amounts of data and improve their performance over time through experience.


In [52]:
from langchain.messages import HumanMessage, AIMessage,SystemMessage
model=init_chat_model('openai:gpt-3.5-turbo')

messages = [
    SystemMessage("You are a python developer"),
    HumanMessage("Please help me with the top 3 interview question related to AI field")
]

response=model.invoke(messages)
print(response.content)

Certainly! Here are three common interview questions related to the field of AI:

1. What is the difference between supervised and unsupervised learning?
   - Supervised learning is a type of machine learning where the model is trained on labeled data, which means that both the input data and the corresponding output labels are provided during training. In unsupervised learning, the model is trained on unlabeled data, and it needs to find patterns or structures in the data without explicit guidance.

2. Can you explain the concept of overfitting in machine learning?
   - Overfitting occurs when a machine learning model performs well on the training data but fails to generalize to unseen data. This happens when the model is too complex and captures noise in the training data rather than the underlying patterns. Overfitting can be addressed by using techniques like regularization, cross-validation, and early stopping.

3. How does a neural network work?
   - A neural network is a type of

In [53]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage
system_msg= SystemMessage("""You are a senior python developer with expertise in web frameworks.
always provide code examples and explain you reasoning. Be concisebut thorough in your explanation""")

message = [system_msg, HumanMessage('How do i create s REST API')]

response= model.invoke(message)
print(response.content)


To create a REST API in Python, you can use a web framework like Flask or Django. Here's a simple example using Flask:

```python
from flask import Flask, jsonify

app = Flask(__name__)

@app.route('/api/endpoint', methods=['GET'])
def get_data():
    data = {'key': 'value'}
    return jsonify(data)

if __name__ == '__main__':
    app.run(debug=True)
```

In this example:

1. We define a Flask application and create an endpoint `/api/endpoint` using the `@app.route` decorator.
2. The `get_data` function returns a JSON response using Flask's `jsonify` function.
3. We run the Flask application using `app.run()`.

You can test this API by running the script and sending a GET request to `http://localhost:5000/api/endpoint`. Make sure to install Flask using `pip install flask`.

Remember to add more endpoints and logic to your API as needed. Flask provides flexibility in structuring your routes and handling requests.


In [55]:
# Metadata

human_msg = HumanMessage(
    content="Hello!",
    name="alice",
    id="msg_123"
)

response=model.invoke([human_msg])
response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Ebh8veKoysxIrJpxCzKedOKF5yGbj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1626d-2a50-703a-95f7-597676907937-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [56]:
from langchain.messages import AIMessage,HumanMessage,SystemMessage
ai_msg = AIMessage("I am happy to help you with you question")

messages = [SystemMessage("You are a helpful assistant"),
            HumanMessage("Can you help me"),
            ai_msg,]
response=model.invoke(messages)
response.content


"Sure, I'll do my best to assist you. What do you need help with?"

In [57]:
response.usage_metadata

{'input_tokens': 33,
 'output_tokens': 18,
 'total_tokens': 51,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}